# Tutorial 5 — Fine-Tuning

**Course:** Text & Language Processing / LLM Practical Track
**Format:** Hands-on notebook (Colab-ready; a GPU helps but is not required)
**Suggested duration:** 120–150 minutes
**Prerequisites:** Tutorial 0 (Ecosystem Tour), Tutorial 1 (Tokenization), Tutorial 2 (Internals)

Everything so far used models as they were shipped. This notebook changes their weights.

By the end you will be able to:

- decide whether a problem actually calls for fine-tuning, or for prompting or retrieval instead;
- prepare a dataset: tokenize it once, batch it efficiently, and keep the label mapping straight;
- attach a fresh **classification head** and explain why it starts out random;
- train with **`Trainer`**, and read what the loss curve is telling you;
- measure **before and after** on a held-out split, so the improvement is evidence rather than hope;
- save a model and reload it through `pipeline()`;
- fine-tune a large model on one GPU with **LoRA**, and say what it trains and what it freezes.

## 0. Mental model: three ways to change a model's behaviour

Fine-tuning is the expensive option. It is worth knowing when the cheap ones suffice.

| Approach | What changes | Costs | Right when |
|---|---|---|---|
| **Prompting** | Nothing. Instructions only. | Tokens per call | The model can already do it, given clear instructions |
| **RAG** | Nothing. Context is retrieved. | Tokens + an index | It lacks *knowledge*, and that knowledge changes |
| **Fine-tuning** | The weights | GPU time + data + a model to host | It lacks a *skill*, format, or domain style |

The distinction that saves the most wasted effort: **fine-tuning teaches behaviour, not facts.** If your
model needs to know your company's current pricing, fine-tuning is the wrong tool — the prices change and
the weights do not. If it needs to reliably emit your schema, classify your domain's text, or write in a
particular register, that is a skill, and fine-tuning is the right tool.

What actually happens is ordinary supervised learning:

```text
batch → forward pass → loss vs labels → backward pass → optimizer updates weights
```

Tutorial 2 covered the forward pass. Fine-tuning adds the three steps after it, repeated a few thousand
times. Nothing more mysterious than that.

**Full fine-tuning vs LoRA.** Updating every weight of a 7B model needs memory for the weights, their
gradients, and optimizer state — comfortably over 80 GB. LoRA (section 9) freezes the original weights and
trains small adapter matrices instead, usually under 1% of the parameters, which is what makes fine-tuning
possible on one consumer GPU.

## 1. Install the libraries

In [ ]:
!pip -q install -U transformers datasets evaluate accelerate peft scikit-learn

### Code walkthrough — what each adds

`transformers`, `datasets` and `evaluate` you met in tutorial 0. Three are new:

- **`accelerate`** — `Trainer` uses it underneath to place tensors on whatever hardware exists. You do not
  call it directly; without it installed, `Trainer` refuses to start.
- **`peft`** — Parameter-Efficient Fine-Tuning, the library providing LoRA in section 9.
- **`scikit-learn`** — needed by `evaluate` for F1 and precision/recall.

## 2. The task and the data

We fine-tune a sentiment classifier on `rotten_tomatoes` — movie reviews labelled positive or negative. It is
small enough to train in minutes and real enough to behave like a genuine dataset.

Tutorial 0 evaluated an off-the-shelf model on this data. Now we train our own and compare.

In [ ]:
from datasets import load_dataset

RAW = load_dataset("cornell-movie-review-data/rotten_tomatoes")
print(RAW)

labels = RAW["train"].features["label"].names
id2label = {i: name for i, name in enumerate(labels)}
label2id = {name: i for i, name in enumerate(labels)}
print("\nlabels:", id2label)

train_ds = RAW["train"].shuffle(seed=0).select(range(2000))
eval_ds  = RAW["test"].shuffle(seed=0).select(range(500))

import collections
print("train balance:", dict(collections.Counter(train_ds["label"])))
print("eval balance: ", dict(collections.Counter(eval_ds["label"])))
print()
print(train_ds[0])

### Code walkthrough — the dataset id, and why we shuffle

**The fully-qualified id matters.** `load_dataset("rotten_tomatoes")` — the bare name — fails on current
versions of `huggingface_hub`, which require `namespace/name`. Use
`cornell-movie-review-data/rotten_tomatoes`.

**`.shuffle(seed=0)` before `.select(...)` is not optional here.** This dataset stores each split sorted by
label: all positives, then all negatives. Take `train[:2000]` without shuffling and you get 2,000 positive
reviews and a model that learns to answer "positive" to everything. The balance printout is there so you
catch that class of mistake by looking rather than by debugging — check it on every dataset you train on.

**`id2label` / `label2id`** come straight from the dataset's `ClassLabel` (tutorial 1, section 4). Building
them from `.names` rather than hard-coding `{0: "negative", 1: "positive"}` means the mapping cannot drift
from the data. They get attached to the model in section 4, so the saved model carries its own label names.

2,000 training rows is small on purpose — a few minutes on a CPU. Scale it up once the pipeline works.

## 3. Tokenize once, pad per batch

The model needs `input_ids`, not strings. Tokenizing the whole dataset up front is far faster than doing it
per batch, but padding up front would be wasteful — so we tokenize now and pad later, per batch.

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

CHECKPOINT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
eval_tok  = eval_ds.map(tokenize, batched=True, remove_columns=["text"])

collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_tok)
print()
lengths = [len(x) for x in train_tok["input_ids"]]
print(f"token lengths: min {min(lengths)}, median {sorted(lengths)[len(lengths)//2]}, max {max(lengths)}")

### Code walkthrough — `map`, and dynamic padding

**`.map(tokenize, batched=True)`** applies the function to chunks of rows rather than one at a time, which
lets the fast Rust tokenizer work on many texts per call — typically an order of magnitude quicker. The
result is cached on disk, so re-running the cell is instant.

**`remove_columns=["text"]`** drops the raw strings once they are tokenized. `Trainer` passes every remaining
column to the model, and a column of strings would raise. Keep `label` — the model expects it by that name.

**No padding here.** `truncation=True, max_length=256` caps the long reviews, but short ones stay short.
Padding everything to 256 now would mean computing over mostly padding for the whole run.

**`DataCollatorWithPadding`** does the padding at batch-assembly time instead, to the longest sequence *in
that batch* — tutorial 1's `padding="longest"`, applied per batch. Since we shuffled, batches contain mixed
lengths and the saving is real; sorting by length would save more, at the cost of correlated batches.

The collator also builds the `attention_mask`, which is what stops the model attending to padding — the
silent-corruption failure from tutorial 2, section 9.

## 4. A model with a fresh head

`AutoModelForSequenceClassification` loads DistilBERT's pretrained body and attaches a **new classification
head** on top. The body knows English; the head knows nothing at all.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

total = sum(p.numel() for p in model.parameters())
head  = sum(p.numel() for n, p in model.named_parameters()
            if n.startswith(("classifier", "pre_classifier")))
print(f"total parameters: {total:,}")
print(f"classifier head:  {head:,}  ({head/total:.2%})")

### Before you run

Loading that model prints a warning, and it is the one warning in this notebook you should be glad to see:

> *Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint ...
> You should probably TRAIN this model on a down-stream task.*

It means exactly what it says. `distilbert-base-uncased` was pretrained on masked language modelling — it has
no classification head, so one was created with **random weights**. Before training, this model's predictions
are noise.

That is why section 6 measures accuracy *before* training. On a balanced two-class problem the untrained head
should score around 50%: chance. If it scores far from chance, something is wrong with the labels.

The warning is only alarming when it names weights you expected to be pretrained — a sign of a checkpoint
mismatch rather than a fresh head.

## 5. Metrics

`Trainer` computes loss on its own, but loss is hard to interpret. `compute_metrics` turns raw predictions
into numbers you can report.

In [ ]:
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, refs = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=refs)["accuracy"],
        "f1": f1.compute(predictions=preds, references=refs, average="macro")["f1"],
    }

### Code walkthrough — from logits to metrics

`Trainer` hands this function a tuple of `(logits, labels)` as NumPy arrays. The logits are
`[n_examples, n_labels]` — one score per class per example, exactly tutorial 2's section 5 output.

**`np.argmax(logits, axis=-1)`** picks the highest-scoring class per row. No softmax is needed: softmax is
monotonic, so it cannot change which class wins.

**`average="macro"`** on F1 averages the per-class scores with equal weight, rather than weighting by class
frequency. On a balanced dataset the two barely differ; on an imbalanced one, macro-F1 is the honest number —
it refuses to be flattered by a majority class the model always guesses. Report accuracy *and* macro-F1 by
default and the two together will tell you when something is off.

## 6. Train

`TrainingArguments` holds the hyperparameters; `Trainer` runs the loop. We evaluate explicitly before and
after rather than on a schedule, which keeps the arguments minimal and portable across library versions.

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="./sentiment-distilbert",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=25,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

before = trainer.evaluate()
print("BEFORE:", {k: round(v, 4) for k, v in before.items() if k in ("eval_accuracy", "eval_f1")})

In [ ]:
trainer.train()

after = trainer.evaluate()
print()
print("BEFORE:", {k: round(v, 4) for k, v in before.items() if k in ("eval_accuracy", "eval_f1")})
print("AFTER: ", {k: round(v, 4) for k, v in after.items()  if k in ("eval_accuracy", "eval_f1")})

### Code walkthrough — the hyperparameters that matter

**`learning_rate=2e-5`** is the one to get right. Fine-tuning learning rates are 100–1000× smaller than
training-from-scratch ones, because the weights are already good and large steps destroy what pretraining
built. `2e-5` to `5e-5` is the standard band for BERT-family models. Set it to `1e-3` and you will watch the
model get worse — an instructive experiment worth running once.

**`num_train_epochs=2`** is deliberately low. With 2,000 examples, more epochs overfit quickly: training loss
keeps falling while eval accuracy stalls or drops. That divergence, not the training loss alone, is the
signal to watch in the log.

**`per_device_train_batch_size=16`** is the first thing to cut on an out-of-memory error. To keep the
*effective* batch size while using less memory, add `gradient_accumulation_steps=2` — it accumulates
gradients over two batches before stepping.

**`weight_decay=0.01`** is mild regularisation. **`report_to="none"`** stops `Trainer` trying to log to
Weights & Biases; drop it if you want that.

**`logging_steps=25`** prints training loss every 25 steps. Read those numbers: a loss that plateaus
immediately usually means the learning rate is too low, or the labels are wrong.

A note on portability — `Trainer`'s argument names have shifted across versions (`evaluation_strategy` became
`eval_strategy`; `tokenizer=` became `processing_class=`). The arguments used here are stable ones, which is
why the collator carries the tokenizer rather than passing it to `Trainer` directly.

## 7. Use the model

Save it, reload it, and run it through `pipeline()` — the same interface tutorial 0 opened with, now pointed
at a model you trained.

In [ ]:
from transformers import pipeline

SAVE_DIR = "./sentiment-distilbert/final"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

classifier = pipeline("sentiment-analysis", model=SAVE_DIR)

for text in [
    "A tender, beautifully acted film that earns every one of its tears.",
    "Ninety minutes I will never get back.",
    "It has moments, but the pacing undoes most of them.",
]:
    out = classifier(text)[0]
    print(f"{out['label']:>8}  {out['score']:.3f}  {text[:55]}")

### Code walkthrough — why the labels come out readable

**`save_model`** writes the weights and config; **`tokenizer.save_pretrained`** writes the tokenizer beside
them. Save only the model and you have created tutorial 1's pitfall 3 for your future self — a checkpoint
whose tokenizer must be guessed.

The pipeline prints `positive` / `negative` rather than `LABEL_0` / `LABEL_1` because `id2label` was passed
in section 4 and travelled into the saved config. It costs one argument at load time and saves every
consumer of the model from looking the mapping up.

`SAVE_DIR` is a local path here. `trainer.push_to_hub()` would put it on the Hub instead, where
`AutoTokenizer`/`AutoModel` could load it by name from anywhere — the upload direction of tutorial 0's
section 2.

### Exercise 7.1

Compare your model against `distilbert-base-uncased-finetuned-sst-2-english`, the off-the-shelf model from
tutorial 0, on the *same* `eval_tok` split. SST-2 and rotten_tomatoes are both movie-review sentiment, so
this is a mild domain shift.

Does 2,000 in-domain examples beat a model trained on far more out-of-domain ones? Try again with 500
training examples, and with 8,000. The curve of "accuracy against training-set size" is the most useful plot
in applied machine learning, and it tells you whether labelling more data is worth the money.

## 8. Reading the training log

Three patterns account for most of what goes wrong, and all three are visible in the numbers `Trainer`
prints.

**Training loss falls, eval accuracy rises.** Working as intended. Keep going while both hold.

**Training loss falls, eval accuracy flat or falling.** Overfitting: the model is memorising these 2,000
reviews rather than learning sentiment. Fix with fewer epochs, more data, or more regularisation. This is the
most common outcome on small datasets, and the reason a held-out split is non-negotiable.

**Training loss barely moves.** Something is broken rather than merely suboptimal. In rough order of
likelihood: the learning rate is far too low; labels are misaligned with inputs; the text column was dropped
or is empty; or everything is being truncated to nothing. Check a decoded batch before touching
hyperparameters:

```python
batch = collator([train_tok[i] for i in range(4)])
print(tokenizer.decode(batch["input_ids"][0]))
print(batch["labels"][:4])
```

That one cell catches more real bugs than any amount of hyperparameter tuning.

## 9. LoRA: fine-tuning a large model on one GPU

Full fine-tuning updates every weight, and needs memory for the weights, their gradients, and optimizer
state — roughly 12–16 bytes per parameter. That is fine for DistilBERT's 67M and impossible for 7B on a
single consumer card.

**LoRA** (Low-Rank Adaptation) freezes the pretrained weights and inserts small trainable matrices beside
them. Instead of learning a full update `ΔW`, it learns `ΔW = B·A`, where `A` and `B` are much thinner — rank
`r` rather than full width. Typically under 1% of parameters end up trainable.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

base = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT, num_labels=2, id2label=id2label, label2id=label2id,
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
)

lora_model = get_peft_model(base, lora_config)
lora_model.print_trainable_parameters()

### Code walkthrough — the LoRA knobs

**`r=8`** is the rank, and the main capacity knob. Higher `r` means more trainable parameters and more
ability to fit, at more memory. 8–64 covers most cases; start at 8 and raise it only if the model underfits.

**`lora_alpha=16`** scales the adapter's contribution by `alpha/r`. The common convention is `alpha = 2r`,
which is what is used here. Treat the pair together rather than tuning each in isolation.

**`target_modules=["q_lin", "v_lin"]`** chooses which layers get adapters — here DistilBERT's attention query
and value projections. Query and value are the usual choice, and the names are architecture-specific
(`q_proj`/`v_proj` on Llama, `c_attn` on GPT-2). Print `base` to find them for an unfamiliar model.

**`task_type`** tells PEFT which head to keep trainable. For `SEQ_CLS` the classification head trains in full
alongside the adapters, which it must — it is random, and a rank-8 adapter cannot rescue a random head.

`print_trainable_parameters()` reports the payoff, usually well under 1%. From here, `lora_model` goes into
the same `Trainer` as before; nothing else in the loop changes.

**Two practical consequences.** Adapters are tiny — a few megabytes — so you can keep many task-specific
adapters for one base model and swap them at load time. And for the memory win to be decisive on large
models, combine LoRA with 4-bit quantization of the frozen base (**QLoRA**, via `bitsandbytes`), which is
what makes 7B fine-tuning fit on a 16 GB card.

## 10. Pitfalls

**Evaluating on data you trained on.** The cardinal sin. Keep the test split untouched until the end; tune on
a validation split if you need one.

**An unshuffled, sorted dataset.** Section 2's trap. Always print the class balance of what you actually
selected.

**Class imbalance read as success.** 95% accuracy on a 95/5 split can mean the model learned to say
"majority". Macro-F1, or a confusion matrix, will say so.

**A learning rate from a different era.** `1e-3` is reasonable when training from scratch and destructive
when fine-tuning. Stay near `2e-5` for BERT-family models; LoRA tolerates more, often `1e-4`.

**Losing the tokenizer.** Save it with the model, every time.

**Catastrophic forgetting.** Fine-tuning hard on a narrow task degrades general ability. If you need both,
you need a mixed dataset — or LoRA adapters you can detach.

### Mini-lab

Fine-tune for a task of your own, end to end:

1. pick a classification dataset that is *not* sentiment — `ag_news` (topics) or `emotion` (six classes) are
   good, and non-English data is more interesting still;
2. check the class balance and shuffle before slicing;
3. train a baseline with the defaults here;
4. measure before and after on a held-out split, reporting accuracy **and** macro-F1;
5. change exactly one thing — learning rate, epochs, training-set size, or LoRA instead of full fine-tuning —
   and measure again;
6. write two sentences: what changed, and was it worth the compute?

Step 5 is the point. One controlled change with a measurement beats ten simultaneous ones.

## Glossary — quick reference

| Term | Meaning |
|---|---|
| **Fine-tuning** | Continuing training on a new task, updating pretrained weights. Teaches skills, not facts. |
| **Classification head** | A small layer on top of the body mapping hidden states to class scores. Starts random. |
| **`DataCollatorWithPadding`** | Pads each batch to its own longest sequence at assembly time, and builds the attention mask. |
| **Epoch** | One pass over the training set. |
| **Learning rate** | Step size. Fine-tuning wants ~`2e-5`, far below from-scratch values. |
| **Gradient accumulation** | Summing gradients over several batches before stepping, for a larger effective batch on fixed memory. |
| **Overfitting** | Training loss falling while eval metrics stall or worsen. |
| **Macro-F1** | F1 averaged with equal weight per class; the honest metric under imbalance. |
| **LoRA** | Freezing pretrained weights and training small low-rank adapters instead. |
| **Rank (`r`)** | LoRA's capacity knob — adapter width, and its parameter count. |
| **QLoRA** | LoRA over a 4-bit quantized frozen base; what fits 7B fine-tuning on one consumer GPU. |
| **Catastrophic forgetting** | Losing general ability while specialising hard on a narrow task. |

---